# 알약 객체 탐지 — Colab에서 전 과정 해 보기

GPU가 없는 컴퓨터에서 **전처리 → EDA → 학습 → 평가 → 앙상블 → `submission.csv`** 를 끝까지 해 보는 노트북입니다.

실제 작업은 이 노트북이 아니라 **브라우저 화면**에서 합니다. 여기 셀들은 그 화면을 띄우는 데까지가 일이고, 그 뒤로는 무엇을 누를지 안내합니다.

## 준비물

| 무엇 | 왜 |
| --- | --- |
| **NVIDIA GPU runtime** | DINO detector는 `device="cuda"`가 아니면 시작을 거부합니다 |
| **팀 AWS 자격 증명과 bucket 이름** | 데이터와 checkpoint가 전부 팀 S3에 있습니다 |

자격 증명이 없다면 이 노트북 대신 `docs/reproduce.md`를 보세요. 공개 번들로 자격 증명 없이 최고 점수를 재현합니다.

## 걸리는 시간

설치와 빌드에 10~15분, 그 뒤는 무엇을 하느냐에 달렸습니다. **학습 12 epoch은 진짜 판에서 하루 가까이 걸립니다.** 시연이라면 `epochs`를 1~2로 낮추세요.

## 0. GPU runtime 고르기

`런타임 > 런타임 유형 변경`에서 **NVIDIA GPU**를 고릅니다.

이걸 빠뜨리면 아래가 전부 성공하고 **학습 시작 직전에** 실패합니다. 아래 셀로 먼저 확인하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 저장소와 dependency

`torchaudio`를 먼저 지우는 이유는 Colab에 미리 깔린 것이 `requirements.txt`의 torch version과 짝이 맞지 않아 설치를 되돌리기 때문입니다.

In [ ]:
%cd /content
!git clone https://github.com/LittleBitAI/codeit-beginner-project.git
%cd codeit-beginner-project
!python -m pip install --upgrade pip setuptools wheel
!python -m pip uninstall -y torchaudio
!python -m pip install -r requirements.txt

### 여기서 런타임을 재시작합니다

`런타임 > 세션 다시 시작`을 누르세요. Colab에 미리 깔린 torch가 메모리에 남아 있어, 재시작하지 않으면 방금 설치한 version이 잡히지 않습니다.

재시작한 뒤에는 **아래 셀부터** 이어서 실행하면 됩니다. 위 셀들을 다시 돌릴 필요는 없습니다.

In [ ]:
%cd /content/codeit-beginner-project
!python onboarding/scripts/verify_onboarding.py --profile colab

Python version, package, CUDA, 실제 tensor 연산까지 확인합니다. 실패하면 어느 단계인지 알려 주므로 그것부터 고칩니다.

`mmcv._ext`가 통과해야 detector를 만들 수 있습니다. 여기서 막히면 아래 학습 단계는 시도해도 소용없습니다.

## 2. 팀 S3 자격 증명

**키를 셀에 직접 적지 마세요.** 노트북을 저장하면 그 값이 파일에 그대로 남고, 세션을 삭제해도 지워지지 않습니다. 왼쪽 열쇠 아이콘(**보안 비밀**)에 넣으면 노트북에는 이름만 남습니다.

넣을 이름 세 개: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `PILL_STORAGE_S3_BUCKET`. 각각 `노트북 액세스 권한`을 켜야 아래 셀이 읽습니다.

`PILL_WEB_STATE_WORKSPACE`는 **런타임이 끊겨도 다음 런타임에서 이어서 할 수 있게** 화면 상태를 S3의 자기 자리에 함께 두는 이름입니다. Colab은 언제든 끊기므로 꼭 정하세요.

이름 규칙이 있습니다 — **영문·숫자로 시작하고 `.` `_` `-` 만 쓸 수 있는 64자 이내**입니다. 그대로 S3 key의 한 조각이 되기 때문입니다. 한글이나 공백, `<` `>` 가 들어가면 서버가 거부합니다.

In [ ]:
import os
import re

from google.colab import userdata

# 여기를 본인 이름으로 바꾸세요. 안 바꾸면 아래에서 멈춥니다 - 두 사람이 같은 이름을
# 쓰면 S3의 같은 자리에 서로의 작업 기록을 덮어씁니다.
# 영문·숫자로 시작하고 . _ - 만 쓸 수 있습니다(한글·공백·<> 는 서버가 거부합니다).
WORKSPACE = "colab-yourname"

if WORKSPACE == "colab-yourname":
    raise SystemExit("WORKSPACE를 본인 이름으로 바꾸세요. 예: colab-hyunwoo")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", WORKSPACE):
    raise SystemExit(f"WORKSPACE 이름이 규칙에 맞지 않습니다: {WORKSPACE!r}")

# 보안 비밀에서만 읽습니다. 여기 값을 적으면 저장된 노트북에 그대로 남습니다.
for name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "PILL_STORAGE_S3_BUCKET"):
    try:
        os.environ[name] = userdata.get(name)
    except Exception as error:
        raise SystemExit(
            f"보안 비밀 {name}을(를) 읽지 못했습니다: {error}\n"
            "왼쪽 열쇠 아이콘에서 이 이름으로 값을 넣고 노트북 액세스 권한을 켜세요."
        )

os.environ["AWS_DEFAULT_REGION"] = "ap-northeast-2"
os.environ["PILL_STORAGE_BACKEND"] = "s3"
os.environ["PILL_WEB_STATE_WORKSPACE"] = WORKSPACE

print("자격 증명과 workspace 준비 완료:", WORKSPACE)

## 3. S3에 닿는지 확인

여기서 dataset 목록이 나오면 자격 증명과 bucket이 맞는 것입니다. 화면을 띄우고 나서 실패하는 것보다 지금 아는 편이 낫습니다.

In [ ]:
!python scripts/make_colab_config.py --list-datasets

## 4. 화면 빌드

React frontend를 한 번 빌드합니다. 3~5분 걸리고, 런타임이 살아 있는 동안 다시 할 필요는 없습니다.

In [ ]:
!node --version
!cd src/pipelines/web/frontend && npm ci && npm run build

빌드가 Node version 때문에 실패하면 아래 셀로 22를 깔고 위 셀을 다시 실행하세요. 필요 없으면 건너뜁니다.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs
!node --version

## 5. 화면 띄우기

서버는 뒤에서 계속 돌아야 하므로 셀을 붙잡지 않게 띄우고, 로그는 파일로 뺍니다. 로그를 화면에 그대로 흘리면 출력이 쌓여 서버가 멈춥니다.

In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PORT = 8000
log_path = Path("artifacts/web/server.log")
log_path.parent.mkdir(parents=True, exist_ok=True)

# sys.executable을 쓰는 이유는 재시작 뒤 PATH의 python이 방금 설치한 환경과 다를 수
# 있기 때문입니다. 다르면 여기서 import부터 실패합니다.
server = subprocess.Popen(
    [sys.executable, "-m", "src.pipelines.web.server", "--port", str(PORT)],
    stdout=log_path.open("w", encoding="utf-8"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    if server.poll() is not None:
        print("서버가 죽었습니다. 로그:")
        print(log_path.read_text(encoding="utf-8"))
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2)
        print("서버가 떴습니다.")
        break
    except Exception:
        time.sleep(1)
else:
    print("아직 응답이 없습니다. 로그:")
    print(log_path.read_text(encoding="utf-8"))

In [ ]:
from google.colab import output

# 노트북 안에 그대로 띄웁니다. 새 창으로 여는 serve_kernel_port_as_window는
# 브라우저 보안 변경으로 막혀 있어 쓰지 않습니다(docs/colab.md에 같은 기록이 있습니다).
output.serve_kernel_port_as_iframe(PORT, height="900")

화면이 노트북 안에 뜹니다. 좁으면 위 셀의 `height`를 키우세요.

**셀 출력이 지워지면 화면도 사라집니다.** 그때는 위 셀만 다시 실행하면 됩니다 — 서버는 뒤에서 계속 돌고 있으므로 학습이 끊기지는 않습니다.

## 6. 화면에서 할 일

왼쪽 목록을 위에서부터 따라갑니다. 각 단계는 끝나야 다음이 됩니다.

### 1) 데이터 준비

| 칸 | 값 |
| --- | --- |
| 학습 : 검증 비율 | `8:2` |
| 원본 경로 | `datasets/pill_detection/raw/v5/original/` |
| Seed | `42` |
| 참조 crop 은행도 만들기 | **켜기** |

나누는 방식은 화면에 칸이 없습니다 — data pipeline이 `group`으로 나눕니다. 같은 조합의 이미지가 학습과 검증으로 갈리지 않게 묶어서 나누는 방식이고, 갈리면 검증 점수가 실제보다 높게 나옵니다.

**crop 은행은 처음부터 켜 두세요.** 6단계 재순위에 필요합니다. 나중에 같은 폴더에 더할 수도 있지만 `이미 있으면 덮어쓰기`를 함께 켜야 하고, 준비를 통째로 다시 도는 만큼 시간이 듭니다.

### 2) EDA

준비한 판을 그대로 읽어 class 분포와 상자 크기를 잽니다. 새로 계산하는 것이 아니라 방금 만든 판을 보는 것이라 금방 끝납니다.

### 3) 새 실험 (학습)

위쪽에 `점수를 받은 설정 채우기 · 최고 점수 detector` 버튼이 있으면 누르세요. Kaggle 0.62437을 받은 설정이 한 번에 채워집니다. 없으면 아래 값을 넣습니다.

| 칸 | 값 | | 칸 | 값 |
| --- | --- | --- | --- | --- |
| model | `dino` | | backbone | `resnet50` |
| input_size | `1280` | | augmentation | `pill_basic` |
| optimizer | `AdamW` | | learning_rate | `0.0001` |
| weight_decay | `0.0001` | | precision | `amp` |
| device | `cuda` | | batch_size | `1` |
| gradient_accumulation_steps | `4` | | num_workers | `0` |
| lr_scheduler | `cosine` | | lr_warmup_steps | `1200` |
| lr_min_factor | `0.01` | | seed | `42` |

**시연이라면 `epochs`를 1~2로 낮추세요.** 12 epoch은 하루 가까이 걸립니다.

A100처럼 메모리가 넉넉한 GPU라면 `batch_size` 2에 `gradient_accumulation_steps` 2로 두면 더 빠릅니다. 유효 batch는 두 값을 곱한 값이라 그래야 위 설정과 같은 4가 됩니다.

### 4) 평가

끝난 실행을 평가합니다. test manifest가 있으면 `submission.csv`가 함께 만들어지고 `내려받기`로 받을 수 있습니다. **여기까지만 해도 제출은 됩니다.**

### 5) 임베딩 학습

1단계에서 만든 crop 은행으로, 잘라 낸 알약이 어떤 class인지 재는 자를 만듭니다. backbone을 바꿔 가며 여럿 만들면(resnet18/34/50) 6단계에서 함께 쓸 수 있습니다.

학습 대기열을 detector와 함께 쓰므로 **detector 학습이 도는 중에는 시작하지 않습니다.**

### 6) 앙상블

- **모델** — 끝난 실행 둘 이상을 골라 WBF로 합칩니다.
- **임베딩** — 5단계 임베딩으로 상자 점수를 다시 매깁니다. 상자와 class는 그대로 두고 점수만 바꿉니다.

합치기 전에 후보들이 얼마나 닮았는지 진단해 줍니다. **약한 실행을 넣으면 점수가 내려갑니다.**

| 구성 | Kaggle |
| --- | --- |
| 7개 전부 합치기 | 0.62087 |
| 단독 최고 | 0.62437 |
| 상위 3개 합치기 | 0.62645 |
| 상위 3개 + 임베딩 재순위 | **0.63594** |

마지막 줄이 이 팀의 최고 점수이고, 여기까지가 전 과정입니다.

로컬 검증 점수는 Kaggle 점수를 예측하지 못합니다. 독립 실험 셋이 모두 무관하거나 반대로 움직였으니, 화면의 mAP만 보고 제출을 고르지 마세요.

## 7. 런타임이 끊겼다면

Colab은 시간이 지나거나 창을 닫으면 런타임을 회수합니다. 학습 중이었다면 그 시점까지의 checkpoint는 S3에 있습니다.

1. 이 노트북을 처음부터 다시 실행합니다(1~5번).
2. **`PILL_WEB_STATE_WORKSPACE`에 지난번과 같은 이름을 넣습니다.** 이 이름으로 지난 실행 기록과 설정을 되찾습니다.
3. 화면의 실행 목록에서 끊긴 실행에 `이어서 학습` 버튼이 올라와 있으면 누릅니다.

이어서 학습은 **새 이름**을 받습니다(`A` 다음은 `A.2`). 같은 이름으로는 시작할 수 없습니다. `epochs`는 전체 계획이므로, 이미 끝난 실행을 더 돌리려면 지난번보다 큰 값을 넣어야 합니다.

끊긴 실행의 `.<run_id>.partial` 폴더는 **아무도 지우지 않습니다.** 그 학습의 유일한 사본이라 지우는 것은 사람이 정할 일입니다.

## 부록: 화면 없이 명령줄로

같은 일을 셀에서 직접 할 수도 있습니다. 화면이 안 뜰 때나, 학습만 걸어 두고 싶을 때 씁니다.

설정은 손으로 쓰지 말고 `scripts/make_colab_config.py`가 만들게 하세요. 고른 값만 적고 나머지는 train의 기본값에 맡깁니다.

In [ ]:
# 전처리 (판 하나 새로 만들기)
!python -m src.main_pipeline --only data --config configs/prepare.v5.aws.json

# 학습 — 먼저 config를 만들고, 그 다음 그 config로 돌립니다.
# !python scripts/make_colab_config.py --list-datasets
# !python scripts/make_colab_config.py --dataset 목록에서-고른-이름 --epochs 12
# !python -m src.main_pipeline --only train --config artifacts/colab/train.json

## 더 읽을 것

| 문서 | 무엇이 있나 |
| --- | --- |
| `README.md` | 로컬(Windows·Linux)에서 같은 일을 하기 |
| `docs/reproduce.md` | 자격 증명 없이 공개 번들로 최고 점수 재현하기 |
| `docs/colab.md` | Colab에서 명령줄로 학습만 돌리기 |
| `docs/team.md` | 저장소 구조, 소유권, storage, Git 규칙 |